# Phase 11 — Final Evaluation and System Validation

This notebook audits the frozen Phase 10 system. It reads the versioned raw outputs and machine-readable report produced by `app.evaluation.system_evaluation`; it does not call or reimplement production services. Missing provider output remains missing rather than becoming a zero-valued accuracy result.

In [1]:
from pathlib import Path
import json
import pandas as pd

candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next(path for path in candidates if (path / 'reports' / 'final_evaluation_v1.json').exists())
REPORT_PATH = ROOT / 'reports' / 'final_evaluation_v1.json'
GRAMMAR_RAW_PATH = ROOT / 'data' / 'evaluation' / 'grammar_evaluation_results_v1.jsonl'
EXERCISE_RAW_PATH = ROOT / 'data' / 'evaluation' / 'exercise_evaluation_results_v1.jsonl'
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
grammar_raw = [json.loads(line) for line in GRAMMAR_RAW_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
exercise_raw = [json.loads(line) for line in EXERCISE_RAW_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(grammar_raw) == report['evaluation_metadata']['grammar_primary_cases']
assert len(exercise_raw) == report['evaluation_metadata']['exercise_requests']
{'root': str(ROOT), 'evaluation_version': report['evaluation_version'], 'grammar_rows': len(grammar_raw), 'exercise_rows': len(exercise_raw)}

{'root': 'C:\\Users\\leanh\\OneDrive\\Desktop\\code\\finnish_learning_assistant',
 'evaluation_version': 'v1',
 'grammar_rows': 40,
 'exercise_rows': 20}

## Dataset, method, and provider configuration

Positive class for grammar detection: **the sentence contains at least one grammar error**. The 40 grammar cases are frozen, human-authored ground truth. Single-error classification is scored separately from set-based multiple-error classification. Canonical corrections use normalized exact match; valid alternative rewrites require separate qualitative review. Explanation quality uses a reviewer-attributed 1–3 rubric for grammatical correctness, relevance, clarity, and learner usefulness. In this run, that rubric is an AI-assisted qualitative linguistic review under reviewer identity `anhtrunghieule`, not a human/native-teacher assessment. Exercise structure and linguistic quality are deliberately separate.

In [2]:
metadata = report['evaluation_metadata']
pd.DataFrame([
    {'item': 'evaluation date', 'value': metadata['evaluation_date']},
    {'item': 'git commit', 'value': metadata['git_commit']},
    {'item': 'provider', 'value': metadata['llm']['provider']},
    {'item': 'model', 'value': metadata['llm']['model'] or 'not configured'},
    {'item': 'API key configured', 'value': metadata['llm']['api_key_configured']},
    {'item': 'temperature', 'value': metadata['llm']['temperature']},
    {'item': 'grammar dataset SHA-256', 'value': metadata['dataset_hashes']['grammar_cases_v1_sha256']},
    {'item': 'grammar prompt SHA-256', 'value': metadata['prompt_hashes']['grammar_checker_prompt_sha256']},
] )

,item,value
0,evaluation date,2026-09-13T02:23:42Z
1,git commit,81646ba03a6123407e39d08e5c68c082d7040ff8
2,provider,openai_compatible
3,model,gpt-5.6-luna
4,API key configured,True
5,temperature,0.0
6,grammar dataset SHA-256,2e336b2842ebc58ca4604e1fb9c6aa99ca01cac36b72e4...
7,grammar prompt SHA-256,387f859081a4324133a40b8dcf46c241cc05af2b221aa8...


## Grammar correctness detection and false positives

Detection metrics use only cases with actual typed predictions. Provider failures remain visible in the request totals and are not silently treated as true negatives.

In [3]:
grammar = report['grammar']
detection = grammar['detection']
fpr = grammar['false_positive_rate']
pd.DataFrame([
    {'measure': 'evaluated predictions', 'value': detection['evaluated_cases_with_prediction'], 'denominator': detection['total_dataset_cases']},
    {'measure': 'accuracy', 'value': detection['accuracy'], 'denominator': detection['evaluated_cases_with_prediction']},
    {'measure': 'precision (error-positive)', 'value': detection['precision'], 'denominator': detection['evaluated_cases_with_prediction']},
    {'measure': 'recall (error-positive)', 'value': detection['recall'], 'denominator': detection['evaluated_cases_with_prediction']},
    {'measure': 'F1 (error-positive)', 'value': detection['f1'], 'denominator': detection['evaluated_cases_with_prediction']},
    {'measure': 'false-positive rate', 'value': fpr['rate'], 'denominator': fpr['evaluated_valid_sentences']},
    {'measure': 'false positives', 'value': fpr['false_positives'], 'denominator': fpr['evaluated_valid_sentences']},
] )

,measure,value,denominator
0,evaluated predictions,39.000000,40
1,accuracy,0.948718,39
2,precision (error-positive),0.944444,39
3,recall (error-positive),0.944444,39
4,F1 (error-positive),0.944444,39
5,false-positive rate,0.045455,21
6,false positives,1.000000,21


## Error types, corrections, explanations, and structured output

Category statistics are meaningful only when predictions exist; small supports are shown explicitly. Multiple-error sentences use set overlap rather than being forced into a single label. Subjective review fields are reported independently.

In [4]:
single = grammar['single_error_category']
category_table = pd.DataFrame.from_dict(single['per_class'], orient='index')
display(category_table)
pd.DataFrame([
    {'component': 'single-error category', **{key: single[key] for key in ('dataset_cases', 'evaluated_cases', 'exact_match_accuracy', 'macro_precision', 'macro_recall', 'macro_f1')}},
    {'component': 'multiple-error category', **{key: grammar['multiple_error_category'][key] for key in ('dataset_cases', 'evaluated_cases', 'set_exact_match_rate', 'micro_precision', 'micro_recall', 'micro_f1', 'macro_f1')}},
])

,f1,precision,recall,support_in_dataset,support_in_evaluated_cases
AGREEMENT,1.000000,1.0,1.0,2,2
CASE_ERROR,1.000000,1.0,1.0,2,2
NOUN_INFLECTION,1.000000,1.0,1.0,2,2
SPELLING,1.000000,1.0,1.0,2,2
VERB_CONJUGATION,0.666667,1.0,0.5,2,2
WORD_ORDER,0.666667,1.0,0.5,2,2


,component,dataset_cases,evaluated_cases,exact_match_accuracy,macro_precision,macro_recall,macro_f1,set_exact_match_rate,micro_precision,micro_recall,micro_f1
0,single-error category,12,12,0.833333,1.0,0.833333,0.888889,NaN,NaN,NaN,NaN
1,multiple-error category,6,6,NaN,NaN,NaN,0.861111,0.666667,0.9,0.818182,0.857143


In [5]:
display(pd.DataFrame([grammar['correction']['counts']]).fillna(0))
display(pd.DataFrame([grammar['structured_output']]))
display(pd.DataFrame(grammar['explanation_quality']['criterion_summary']).T)
grammar['explanation_quality']['methodology']


,acceptable_alternative,exact_match,incorrect_correction,no_output,not_applicable
0,0,17,1,1,21


,controlled_provider_or_configuration_failures,malformed_or_invalid_responses,provider_responses_received,provider_transport_or_http_failures,success_rate,total_primary_requests,valid_typed_responses
0,0,1,40,0,0.975,40,39


,mean,rated,score_1,score_1_or_2_rate,score_2,score_3,score_3_rate
clarity,2.923077,39.0,0.0,0.076923,3.0,36.0,0.923077
grammatical_correctness,2.666667,39.0,4.0,0.230769,5.0,30.0,0.769231
learner_usefulness,2.717949,39.0,4.0,0.179487,3.0,32.0,0.820513
relevance,2.897436,39.0,1.0,0.076923,2.0,36.0,0.923077


{'dimensions': ['grammatical correctness',
  'relevance',
  'clarity',
  'learner usefulness'],
 'review_method': 'AI-assisted qualitative linguistic review',
 'review_population': 'all successful primary-set grammar responses',
 'scale': '1-3',
 'subjectivity_note': 'Ratings are an AI-assisted linguistic judgment under the named reviewer identity, not a human/native-teacher assessment or objective linguistic truth.'}

## Phase 5 ML baseline verification

This is a different task from live grammar checking: it assigns one of three labels to a controlled synthetic sentence already known to be incorrect. The evaluator reloads the saved pipeline, confirms group-isolated splits, and recomputes every split metric and stored test prediction.

In [6]:
ml = report['ml_baseline']
ml_metrics = pd.DataFrame({split: {key: values[key] for key in ('accuracy', 'macro_f1', 'log_loss')} for split, values in ml['metrics'].items()}).T
display(ml_metrics)
display(pd.DataFrame(ml['metrics']['test']['confusion_matrix']['values'], index=ml['labels'], columns=ml['labels']))
{'all_checks_passed': ml['all_checks_passed'], 'metric_matches': ml['recomputed_metric_matches'], 'group_intersections': ml['split_identity']['actual_group_intersections'], 'shortcut_checks': ml['shortcut_checks']}

,accuracy,macro_f1,log_loss
test,0.710468,0.709202,0.703513
train,0.933365,0.933129,0.509013
validation,0.691111,0.687278,0.705492


,AGREEMENT,CASE_ERROR,VERB_CONJUGATION
AGREEMENT,89,56,4
CASE_ERROR,53,89,8
VERB_CONJUGATION,2,7,141


{'all_checks_passed': True,
 'metric_matches': {'test': True, 'train': True, 'validation': True},
 'group_intersections': {'train_test': 0,
  'train_validation': 0,
  'validation_test': 0},
 'shortcut_checks': {'test_accuracy_by_source': {'UD_Finnish-FTB': 0.7794117647058824,
   'UD_Finnish-TDT': 0.6530612244897959},
  'test_source_only_majority_accuracy': 0.44097995545657015,
  'top_feature_examples': {'AGREEMENT': ['iset ',
    'iset',
    'set ',
    'et ',
    'set'],
   'CASE_ERROR': ['ine', 'nen ', 'inen ', 'inen', 'nen'],
   'VERB_CONJUGATION': [' hän ', 'hän ', ' hän', 'hän', ' hä']},
  'train_test_macro_f1_gap': 0.223927333703556,
  'training_global_majority_accuracy': 0.3336506425511661}}

## Vocabulary, learner profile, and personalization

The vocabulary reference set scores lookup status, lemma, POS, morphology, meaning availability/content where source-backed, provenance, ambiguity, normalization, Unicode, and unknown-word behavior separately. Controlled temporary databases make profile expectations exact and isolate learners.

In [7]:
display(pd.DataFrame(report['vocabulary']['metrics']).T)
display(pd.DataFrame([report['vocabulary_index']]))
display(pd.DataFrame(report['profile']['scenarios']))
display(pd.DataFrame(report['personalization']['profiles']))

,evaluated,passed,rate
ambiguity_behavior_correct,8.0,8.0,1.0
example_relevance,15.0,15.0,1.0
lemma_correct,16.0,16.0,1.0
lookup_status_correct,16.0,16.0,1.0
meaning_correct_or_availability_expected,15.0,15.0,1.0
morphology_correct,15.0,15.0,1.0
pos_correct,16.0,16.0,1.0
provenance_available,15.0,15.0,1.0
unknown_word_behavior_correct,1.0,1.0,1.0


,ambiguous_lookup_keys,analyses_with_meanings,analysis_records,forms_with_multiple_feature_analyses,index_sha256,lookup_keys,manifest_hash_matches,maximum_analyses_for_one_key,normalized_lemmas,scope_note
0,5313,12070,40624,5813,f136908c864ac1de6311400cb3aea2dbc61220397c0798...,93547,True,6,39506,These are statistics for the corpus-bounded Ph...


,actual,expected,scenario,status
0,{'empty_learner': {'learner_id': 'empty_learne...,{'empty_learner': {'learner_id': 'empty_learne...,no_checks,PASS
1,{'correct_learner': {'learner_id': 'correct_le...,{'correct_learner': {'learner_id': 'correct_le...,correct_checks_only,PASS
2,{'one_error_learner': {'learner_id': 'one_erro...,{'one_error_learner': {'learner_id': 'one_erro...,one_error,PASS
3,{'several_learner': {'learner_id': 'several_le...,{'several_learner': {'learner_id': 'several_le...,several_categories,PASS
4,"{'tie_learner': {'learner_id': 'tie_learner', ...","{'tie_learner': {'learner_id': 'tie_learner', ...",tied_categories,PASS
5,"{'learner_a': {'learner_id': 'learner_a', 'pri...","{'learner_a': {'learner_id': 'learner_a', 'pri...",learner_isolation,PASS


,actual_automatic_target,documented_no_history_fallback,expected_automatic_target,explicit_fallback_target,no_history_error,profile,selection_source,status
0,CASE_ERROR,NaN,CASE_ERROR,NaN,NaN,profile_a_case_highest,LEARNER_PROFILE,PASS
1,VERB_CONJUGATION,NaN,VERB_CONJUGATION,NaN,NaN,profile_b_verb_highest,LEARNER_PROFILE,PASS
2,CASE_ERROR,NaN,CASE_ERROR,NaN,NaN,profile_c_tie,LEARNER_PROFILE,PASS
3,NaN,Require an explicit supported topic; CASE_ERRO...,NaN,CASE_ERROR,no supported grammar weakness is available; ch...,profile_d_no_history,NaN,PASS


## Exercises and end-to-end validation

Twenty exercises are requested: four for each of the five supported categories. JSON/schema structure does not stand in for linguistic correctness. The end-to-end checklist separately covers navigation, correct/incorrect/multiple-error flows, persistence, rerun safety, vocabulary, history/profile, practice, answer checking, and controlled failures.

In [8]:
exercise = report['exercise']
display(pd.DataFrame(exercise['by_category']).T)
display(pd.DataFrame([{
    'requested': exercise['requested'],
    'generated': exercise['generated'],
    'schema_valid': exercise['schema_valid'],
    'structurally_valid': exercise['structurally_valid'],
    'structural_rate_all_requests': exercise['structural_validity_rate_over_all_requests'],
    'qualitative_reviews': exercise['manual_review']['completed'],
    'provider_failures': exercise['provider_failures'],
}]))
display(pd.DataFrame(exercise['manual_review']['criterion_summary']).T)
display(pd.DataFrame(report['end_to_end']['scenarios']))


,generated,manually_reviewed,requested,structurally_valid
AGREEMENT,4,4,4,4
CASE_ERROR,4,4,4,4
NOUN_INFLECTION,4,4,4,4
SPELLING,4,4,4,4
VERB_CONJUGATION,4,4,4,4


,requested,generated,schema_valid,structurally_valid,structural_rate_all_requests,qualitative_reviews,provider_failures
0,20,20,20,20,1.0,20,0


,fail,non_pass_rate,partial,pass,pass_rate,rated
answer_correctness,1.0,0.05,0.0,19.0,0.95,20.0
distractor_plausibility,1.0,0.45,8.0,11.0,0.55,20.0
explanation_correctness,1.0,0.15,2.0,17.0,0.85,20.0
no_unrelated_errors,0.0,0.00,0.0,20.0,1.00,20.0
question_clarity,0.0,0.05,1.0,19.0,0.95,20.0
target_relevance,0.0,0.00,0.0,20.0,1.00,20.0
unambiguous,0.0,0.05,1.0,19.0,0.95,20.0


,evidence,scenario,status
0,Streamlit AppTest loaded app.py without an exc...,app_import_and_startup,PASS
1,Six sidebar routes rendered with isolated data...,all_page_navigation,PASS
2,"One explicit action produced one check row, no...",correct_grammar_flow,PASS
3,One parent check and two ordered child errors ...,incorrect_multiple_error_flow,PASS
4,Normal result rendering reruns did not repeat ...,exactly_once_persistence,PASS
5,Persisted errors were immediately visible as e...,history_and_profile_update,PASS
6,Actual local index statuses: {'kouluun': 'foun...,known_ambiguous_unknown_vocabulary,PASS
7,The controlled highest weakness selected a sta...,profile_targeted_practice,PASS
8,Answer checking used the stored exercise and m...,deterministic_answer_checking,PASS
9,Grammar render rerun preserved one insert; pra...,streamlit_rerun_safety,PASS


## Performance, repeatability, failures, and limitations

Local warmed timings and provider/network timings are labeled separately. The repeatability sample is descriptive only. API cost is reported only if actual token/accounting evidence exists.

In [9]:
display(pd.DataFrame(report['performance']).T)
display(pd.DataFrame(report['reliability']['by_case']).T)
display(pd.DataFrame.from_dict(report['failure_analysis']['taxonomy_counts'], orient='index', columns=['count']))
print('API cost:', report['api_cost']['statement'])
print('\nLimitations:')
for item in report['limitations']:
    print('-', item)

,measurement,platform,python,maximum_ms,median_ms,minimum_ms,p95_ms,sample_count
environment,perf_counter wall-clock time in the current lo...,win32,3.12.14,NaN,NaN,NaN,NaN,NaN
exercise_requests,NaN,NaN,NaN,1820.6557,1427.34835,1220.0714,1664.9016,20.0
external_latency_caveat,External API latency depends on the provider a...,External API latency depends on the provider a...,External API latency depends on the provider a...,External API latency depends on the provider a...,External API latency depends on the provider a...,External API latency depends on the provider a...,External API latency depends on the provider a...,External API latency depends on the provider a...
external_provider_latency_measured,True,True,True,True,True,True,True,True
grammar_primary_requests,NaN,NaN,NaN,4135.7867,1873.8147,1285.436,2614.5074,40.0
profile_retrieval,NaN,NaN,NaN,13.2735,10.6813,8.7776,12.5262,30.0
vocabulary_lookup_warmed,NaN,NaN,NaN,0.2938,0.0629,0.0026,0.2891,30.0


""


,count
exercise:ambiguous_answer,1
exercise:incorrect_answer,1
exercise:incorrect_explanation,1
exercise:oversimplified_rule,2
exercise:unclear_question,1
exercise:weak_distractor,9
grammar:analysis_status_mismatch,1
grammar:extra_category,3
grammar:false_negative,1
grammar:false_positive,1


API cost: API cost was not formally measured.

Limitations:
- Grammar explanation quality and exercise linguistic quality use AI-assisted qualitative linguistic review; these ratings are not a human/native-teacher assessment, and automatic structure and correctness metrics do not replace them.
- External LLM behavior is nondeterministic and may change with provider or model updates even when the prompt and dataset hashes remain fixed.
- The Phase 5 baseline uses controlled synthetic examples from only three categories; its scores do not establish real-learner grammar performance.
- Phase 5 character n-grams show suffix, source, and sentence-length shortcut risk, including a substantial train-test performance gap.
- The vocabulary index is bounded by observed UD forms and FinnWordNet overlap; it is neither a complete dictionary nor a complete morphology system.
- Vocabulary meanings are unranked sense candidates and are not context-disambiguated translations.
- The AI-assisted exercise 

## Rubric evidence and project requirements

Statuses are evidence-based: `SATISFIED`, `PARTIAL`, `NOT SATISFIED`, `NOT APPLICABLE`, or `REQUIRES LIVE PRESENTATION`. Presentation and audience criteria that code cannot prove remain explicitly assigned to live evidence.

In [10]:
display(pd.DataFrame(report['rubric_evidence']))
display(pd.DataFrame(report['requirements_audit']))

,evaluation_evidence,implementation_evidence,notes,rubric_criterion,status
0,"Phase 3 output hash and 33,859-record manifest...","Phase 3 parser, processed JSONL, manifest, and...",Original Finnish text is preserved; model-only...,Data transformation before training,SATISFIED
1,All pairwise leakage-group intersections were ...,Grouped train/validation/test JSONL files and ...,Source-group isolation takes priority over exa...,Separate train and test evaluation,SATISFIED
2,"Saved-pipeline metrics were recomputed, includ...","Phase 5 metadata stores train, validation, and...","Log loss is classification log loss, not a neu...",Accuracy and loss for train and test,SATISFIED
3,All 40 live grammar attempts were recorded: 39...,Saved Logistic Regression pipeline and validat...,ML and LLM tasks are different and are not com...,Acceptable model accuracy or prompted LLM func...,SATISFIED
4,End-to-end checks exercise the complete learni...,Project specification addresses recurring Finn...,Audience impact still requires live feedback e...,Real-world/community problem,SATISFIED
5,Controlled persistence/profile/personalization...,Grammar feedback is connected to local history...,Originality is partly a qualitative judgment f...,Original and creative idea,PARTIAL
6,"Artifacts, licenses, service tests, and reprod...","UD, FinnWordNet, Streamlit, SQLite, scikit-lea...",Presentation must still demonstrate the studen...,Technologies beyond curriculum via documentation,SATISFIED
7,All pages rendered and the real Streamlit heal...,Six-page Streamlit UI with centralized routing...,"Automated checks establish function and flow, ...",Logical and aesthetic application/UI library use,SATISFIED
8,Cannot be established through source code or a...,No audience-feedback artifact exists in Phase 11.,Requires live presentation/audience evidence o...,Audience acceptance,REQUIRES LIVE PRESENTATION
9,Not observable in this repository.,Not a software implementation criterion.,Requires live presentation evidence outside Ph...,Confident presentation and audience interaction,REQUIRES LIVE PRESENTATION


,evaluation_evidence,implementation_evidence,limitation,requirement,requirement_id,status
0,real health endpoint and AppTest,app.py,Local MVP only.,Streamlit application launches,MVP-01,SATISFIED
1,E2E correct and incorrect submissions,Grammar Checker form,500-character service limit.,User can enter one Finnish sentence,MVP-02,SATISFIED
2,data/evaluation/grammar_evaluation_results_v1....,GrammarService and prompt,39/40 attempts produced validated outputs; one...,Grammar analysis works,MVP-03,SATISFIED
3,data/evaluation/grammar_evaluation_results_v1....,typed corrected_sentence,"Across 18 erroneous cases, 17 corrections matc...",Sentence correction works,MVP-04,PARTIAL
4,data/evaluation/grammar_evaluation_results_v1....,per-error and overall explanation fields,"AI-assisted review is complete, but four expla...",Grammar explanation works,MVP-05,PARTIAL
5,data/evaluation/grammar_evaluation_results_v1....,seven-category production taxonomy,Single- and multiple-error metrics use small c...,Errors are classified,MVP-06,SATISFIED
6,E2E exactly-once and profile scenarios,SQLite grammar_checks/grammar_errors,Local demo identity; no authentication.,Results are stored,MVP-07,SATISFIED
7,controlled temporary-SQLite profile and person...,ProfileService,Confidence is stored but not used as a weight.,Learner weakness profile updates,MVP-08,SATISFIED
8,data/evaluation/vocabulary_cases_v1.jsonl and ...,Phase 8 local index and service,"Corpus-bounded, not a complete dictionary.",Basic vocabulary lookup works,MVP-09,SATISFIED
9,data/evaluation/exercise_evaluation_results_v1...,ExerciseService,"20/20 generated and passed structure, but one ...",Basic targeted exercises can be generated,MVP-10,PARTIAL


## Verdict

Readiness is mechanical: every approved provider-backed sample and qualitative review must complete, deterministic checks and tests must pass, Streamlit must start, and protected prior-phase artifacts must be unchanged or covered by an explicitly approved Phase 11 bug-fix exception.

In [11]:
print(report['phase_11_verdict'])
for blocker in report['completion_blockers']:
    print('-', blocker)

READY FOR PHASE 12
